**【Colabの基本操作】**

・薄いグレー枠内（「セル」という）がPythonのコード。<br>[　] にポインタをあわせると「▶」マークに変化し、クリックすると枠内のコードが実行される。

・処理が完了すると、セルの左側に「レ」マークが出ますので、それまで待つ。

**【実行手順】**<br>
**１．ランタイムの設定がGPUになっているか確認**<br>
・上のメニューから「ランタイム」を探してクリック<br>
・「ランタイムのタイプを変更」→ 「ハードウェアアクセラレータ」→「T4 GPU」を選択<br>
・「保存」をクリック



**２．環境構築**

**２－１．自分のアノテーションデータを取得**

In [ ]:
# ==============================================================================
# アノテーションデータのアップロードと学習用・検証用への分割
# (受講生の皆さんは、中身を変更せずにそのまま左上の再生ボタンを押してください)
# ==============================================================================

import os
import zipfile
import random
import shutil
from pathlib import Path

VAL_SPLIT_RATIO = 0.2  # 検証用（val）データの割合（0.2 = 20%）
SEED_VALUE = 42        # ランダム分割の再現性を固定するシード値

print("⏳ [1/4] あなたがLabelImgで作成した「画像とtxtが入ったZIPファイル」を選択してください...")
# ブラウザのファイル選択画面が立ち上がります（ゲストモードでも動作します）
uploaded = files.upload()

# アップロードされたZIPファイル名を取得
zip_name = list(uploaded.keys())[0]
extract_dir = 'extracted_data'
output_dir = Path('dataset')

print(f"\n⏳ [2/4] {zip_name} を解凍中...")
if os.path.exists(extract_dir):
    shutil.rmtree(extract_dir)
with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print("\n⏳ [3/4] YOLOv8用のディレクトリ構造を作成中...")
# 既存の古いデータセットがあれば初期化
if output_dir.exists():
    shutil.rmtree(output_dir)

for split in ['train', 'val']:
    (output_dir / 'images' / split).mkdir(parents=True, exist_ok=True)
    (output_dir / 'labels' / split).mkdir(parents=True, exist_ok=True)

print("\n⏳ [4/4] 画像とラベルをシャッフルし、train/val に自動分割中...")
# サポートする画像拡張子
extensions = ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']
all_images = []
for ext in extensions:
    all_images.extend(Path(extract_dir).glob(f'**/{ext}'))

# 画像と対になるアノテーション(txt)が存在するペアのみを抽出（不完全なデータを排除）
valid_pairs = []
for img_path in all_images:
    lbl_path = img_path.with_suffix('.txt')
    if lbl_path.exists():
        valid_pairs.append((img_path, lbl_path))

if len(valid_pairs) == 0:
    raise FileNotFoundError("⚠️ 解凍されたフォルダ内に、画像と対応する .txt ファイルのペアが見つかりませんでした。フォルダ構造を確認してください。")

# 再現性を担保してシャッフル
random.seed(SEED_VALUE)
random.shuffle(valid_pairs)

# 分割位置の計算
val_count = int(len(valid_pairs) * VAL_SPLIT_RATIO)
train_count = len(valid_pairs) - val_count

train_pairs = valid_pairs[val_count:]
val_pairs = valid_pairs[:val_count]

# ファイルのコピー処理（移動ではなくコピーにすることでトラブル時に解凍直後からやり直せる）
def copy_dataset_pairs(pairs, split_name):
    for img_p, lbl_p in pairs:
        # 同名ファイルの衝突を防ぐため、元のフォルダ構造を反映したファイル名にするか、そのままコピーするか
        # ここでは単純にファイル名そのままでコピー（LabelImgの出力がユニークである前提）
        shutil.copy(img_p, output_dir / 'images' / split_name / img_p.name)
        shutil.copy(lbl_p, output_dir / 'labels' / split_name / lbl_p.name)

copy_dataset_pairs(train_pairs, 'train')
copy_dataset_pairs(val_pairs, 'val')

print("\n==================================================")
print("🎉 データセットの準備が正常に完了しました！")
print(f" ┣ 総データ数: {len(valid_pairs)} 組")
print(f" ┣ 学習用 (train): {len(train_pairs)} 組 ({(1 - VAL_SPLIT_RATIO)*100:.0f}%)")
print(f" ┗ 検証用 (val):   {len(val_pairs)} 組 ({VAL_SPLIT_RATIO*100:.0f}%)")
print("==================================================")

⏳ [1/4] Google ドライブからデータセットをダウンロード中...
-> 既にダウンロード済みのZIPファイルを使用します。

⏳ [2/4] ZIPファイルを解凍中...

⏳ [3/4] YOLOv8用のディレクトリ構造を作成中...

⏳ [4/4] 画像とラベルをシャッフルし、train/val に自動分割中...

🎉 データセットの準備が正常に完了しました！
 ┣ 総データ数: 100 組
 ┣ 学習用 (train): 80 組 (80%)
 ┗ 検証用 (val):   20 組 (20%)


**２－２．YAMLファイルを作成**

In [3]:
# ==============================================================================
# AIの教科書目次（YAMLファイル）の自動生成
# ==============================================================================

import yaml

# YAMLファイルに書き込む内容を定義
yaml_data = {
    # さきほど作成したディレクトリの絶対パスを指定（Colab環境用）
    'train': '/content/dataset/images/train',
    'val': '/content/dataset/images/val',

    # AIが分類するクラス数と、その名前（※アノテーションの順序と完全に一致させること）
    'nc': 2,
    'names': ['masked', 'no_mask']
}

# data.yaml という名前でファイルを作成して保存
yaml_file_path = '/content/data.yaml'
with open(yaml_file_path, 'w', encoding='utf-8') as f:
    yaml.dump(yaml_data, f, default_flow_style=False, sort_keys=False)

print(f"🎉 構成ファイル ({yaml_file_path}) を作成しました！")
with open(yaml_file_path, 'r') as f:
    print(f.read())

🎉 構成ファイル (/content/data.yaml) を作成しました！
train: /content/dataset/images/train
val: /content/dataset/images/val
nc: 2
names:
- masked
- no_mask



**２－３．PythorchおよびYOLOの構築**

In [4]:
!pip install ultralytics
from ultralytics import YOLO

**⇒ これで、環境構築は完了**

**３．ベースモデルで推論してみる**

YOLOが公開している機械学習ベースモデル（COCO）は動作速度・予測精度の調整により複数存在します。（80種類のクラスに対応）

速度:速 精度:低 ← YOLOv8n YOLOv8s YOLOv8m YOLOv8l YOLOv8x → 速度:遅 精度:高<br><br>
以下のセルを実行すると、PCカメラのリアルタイム映像をベースモデルで推論します。
いろいろなものを映してみてください。

In [ ]:
import cv2
import numpy as np
import base64
from IPython.display import display, Javascript
from google.colab.output import eval_js
from ultralytics import YOLO

# 1. COCOモデルを読み込む
model = YOLO('yolov8n.pt')

# 2. JavaScriptのWebカメラ制御およびDOM操作コード
def video_stream():
    js = Javascript('''
        var video;
        var div = null;
        var stream;
        var captureCanvas;
        var imgElement;
        var labelElement;

        var shutdown = false;

        function removeDom() {
            shutdown = true;
            if (stream) {
                stream.getTracks().forEach(track => track.stop());
            }
            if (div) {
                div.remove();
            }
        }

        function createDom() {
            if (div !== null) {
                return stream;
            }
            div = document.createElement('div');
            div.style.border = '2px solid black';
            div.style.padding = '3px';
            div.style.width = '100%';
            div.style.maxWidth = '600px';
            document.body.appendChild(div);

            var modelOut = document.createElement('div');
            modelOut.innerHTML = "<span>Status: </span>";
            labelElement = document.createElement('span');
            labelElement.innerText = 'Initializing...';
            labelElement.style.fontWeight = 'bold';
            modelOut.appendChild(labelElement);
            div.appendChild(modelOut);

            video = document.createElement('video');
            video.style.display = 'none'; // 生のカメラ映像は隠す
            video.setAttribute('playsinline', '');
            stream = navigator.mediaDevices.getUserMedia({video: { facingMode: "environment"}});

            imgElement = document.createElement('img');
            imgElement.style.width = '100%';
            imgElement.onclick = () => { removeDom(); }; // クリックで終了
            div.appendChild(imgElement);

            var instruction = document.createElement('div');
            instruction.innerHTML = '<span style="color: red; font-weight: bold;">画像をクリックするとストリームを停止します</span>';
            div.appendChild(instruction);

            captureCanvas = document.createElement('canvas');
            return stream;
        }

        async function getFrame(b64_img) {
            if (shutdown) return null;

            // 初回のみDOM作成とカメラ起動
            if (div === null) {
                stream = await createDom();
                video.srcObject = stream;
                await video.play();
                captureCanvas.width = video.videoWidth;
                captureCanvas.height = video.videoHeight;
            }

            // Python側からアノテーション済みの画像が渡されたらimgタグを更新
            if (b64_img !== "") {
                imgElement.src = b64_img;
                labelElement.innerText = 'Running Inference...';
            }

            // 次のフレームを取得してBase64で返す
            captureCanvas.getContext('2d').drawImage(video, 0, 0);
            return captureCanvas.toDataURL('image/jpeg', 0.8);
        }
    ''')
    display(js)

# 3. PythonとJavaScript間のデータ変換ヘルパー関数
def video_frame(b64_img):
    data = eval_js(f'getFrame("{b64_img}")')
    return data

def js_to_image(js_reply):
    header, encoded = js_reply.split(',', 1)
    decoded = base64.b64decode(encoded)
    img_array = np.frombuffer(decoded, dtype=np.uint8)
    img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
    return img

def image_to_base64(img):
    _, buffer = cv2.imencode('.jpg', img)
    encoded = base64.b64encode(buffer).decode('utf-8')
    return f"data:image/jpeg;base64,{encoded}"

# 4. メインの推論ループ
print("Webカメラを起動します。ブラウザのカメラ許可ダイアログが出たら許可してください。")
video_stream()

b64_img = ""
while True:
    try:
        # JSからカメラ画像を取得しつつ、前回のアノテーション画像をJSに渡して描画
        js_reply = video_frame(b64_img)

        if not js_reply:
            print("ストリームが停止されました。")
            break

        # Base64文字列をOpenCVの画像配列に変換
        frame = js_to_image(js_reply)

        # YOLOv8で推論 (conf=0.5で信頼度50%以上のものを検出、verbose=Falseでログを抑制)
        results = model.predict(frame, conf=0.5, verbose=False)

        # バウンディングボックスが描画された画像を取得
        annotated_frame = results[0].plot()

        # JSに送り返すためにBase64に変換
        b64_img = image_to_base64(annotated_frame)

    except Exception as e:
        print(f"エラー発生により停止しました: {e}")
        break

同じsourceに対しいろいろなモデルで推論をしてみると、実行時間・予測精度に大きな差があると分かります。  
実際にAIの学習をする際はこのモデルを推論したい対象に特化するよう転移学習を行うことが多いため、速度を優先するか精度を優先するか、利用目的によって適切に判断する必要があります。

**３．ファインチューニング**

**３－１． データ構成を確認**<br>
'data.yaml'をダブルクリックして開く。
これは、

・教師データの場所（「パス」という）を指定

・検出対象種類（クラスという）およびクラス名を
指定するための必須ファイルです。

**開いて内容を確認してみましょう**

・'train'はトレーニング、'val'はバリデーションの意味で、これらフォルダ内のデータが教師データとなります。
機械学習の処理は非常に複雑で「ブラックボックス」とも言われますが、あえて簡単に言うと「train」で学習していき「val」で学習がうまくいっているか評価しながら勾配および重みパラメータを調整していきます。
これらのパスが自分の環境でどうなっているか、確かめながら作業してください。
（'test'は推論対象用データを入れたフォルダで学習過程には使われない）


**３－２． FT実行**<br>
開始すると学習の状況表示が更新されていく。<br>
'batch'ぶんのデータが一度に取り込まれ、1stepずつ学習が進行する。そして全データが1回ずつ参照されると'1epoch完了'となる。よってtrain中の全データはepochの値だけ反復して使われるということになる。val中のデータは学習途中あるいは学習終了時にまとめて参照される。
'mAP50-95'が精度の指標であり、この値が上がっていけば学習は順調といえる

In [6]:
# 'data'にyamlファイルパスを指定, 'model'にベースとなるモデルタイプを指定, ほかはそのままで実行
!yolo detect train data=/content/data.yaml model=yolov8n.pt epochs=150 imgsz=512 batch=16 cache=True patience=30 mosaic=0.0 mixup=0.0 fliplr=0.5 flipud=0.0 degrees=10.0 scale=0.5 hsv_h=0.015 hsv_s=0.7 hsv_v=0.4

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=10.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=30, perspectiv

**完了後、所要時間や学習後モデルのサマリーが表示される。**

およその性能を確認してみよう。

【つぎの3つの値を見ておこう】

例）

●150 epochs completed in 0.511 hours.
トレーニング所要時間

●mAP (mean Average Precision)が性能の指標。
基本的にはmAPが高いほど良いモデルで、0から1の間で表される
「誤検出の少なさ」と「検出漏れの少なさ」両方を考慮して算出される

●Speed: 推論処理スピードの参考値。

**成果物として、学習後のモデルファイルが以下のように生成される。パスを確認しておこう**<br>
Optimizer stripped from /content/runs/detect/train/weights/last.pt<br>
Optimizer stripped from /content/runs/detect/train/weights/best.pt



**４．FTモデルで推論**

以下のコードを実行。２．のベースモデル推論とした結果と比べてみよう


In [8]:
import cv2
import numpy as np
import base64
from IPython.display import display, Javascript
from google.colab.output import eval_js
from ultralytics import YOLO

# 1. 学習済みのベストモデルを読み込む
model = YOLO("runs/detect/train/weights/best.pt")

# 2. JavaScriptのWebカメラ制御およびDOM操作コード
def video_stream():
    js = Javascript('''
        var video;
        var div = null;
        var stream;
        var captureCanvas;
        var imgElement;
        var labelElement;

        var shutdown = false;

        function removeDom() {
            shutdown = true;
            if (stream) {
                stream.getTracks().forEach(track => track.stop());
            }
            if (div) {
                div.remove();
            }
        }

        function createDom() {
            if (div !== null) {
                return stream;
            }
            div = document.createElement('div');
            div.style.border = '2px solid black';
            div.style.padding = '3px';
            div.style.width = '100%';
            div.style.maxWidth = '600px';
            document.body.appendChild(div);

            var modelOut = document.createElement('div');
            modelOut.innerHTML = "<span>Status: </span>";
            labelElement = document.createElement('span');
            labelElement.innerText = 'Initializing...';
            labelElement.style.fontWeight = 'bold';
            modelOut.appendChild(labelElement);
            div.appendChild(modelOut);

            video = document.createElement('video');
            video.style.display = 'none'; // 生のカメラ映像は隠す
            video.setAttribute('playsinline', '');
            stream = navigator.mediaDevices.getUserMedia({video: { facingMode: "environment"}});

            imgElement = document.createElement('img');
            imgElement.style.width = '100%';
            imgElement.onclick = () => { removeDom(); }; // クリックで終了
            div.appendChild(imgElement);

            var instruction = document.createElement('div');
            instruction.innerHTML = '<span style="color: red; font-weight: bold;">画像をクリックするとストリームを停止します</span>';
            div.appendChild(instruction);

            captureCanvas = document.createElement('canvas');
            return stream;
        }

        async function getFrame(b64_img) {
            if (shutdown) return null;

            // 初回のみDOM作成とカメラ起動
            if (div === null) {
                stream = await createDom();
                video.srcObject = stream;
                await video.play();
                captureCanvas.width = video.videoWidth;
                captureCanvas.height = video.videoHeight;
            }

            // Python側からアノテーション済みの画像が渡されたらimgタグを更新
            if (b64_img !== "") {
                imgElement.src = b64_img;
                labelElement.innerText = 'Running Inference...';
            }

            // 次のフレームを取得してBase64で返す
            captureCanvas.getContext('2d').drawImage(video, 0, 0);
            return captureCanvas.toDataURL('image/jpeg', 0.8);
        }
    ''')
    display(js)

# 3. PythonとJavaScript間のデータ変換ヘルパー関数
def video_frame(b64_img):
    data = eval_js(f'getFrame("{b64_img}")')
    return data

def js_to_image(js_reply):
    header, encoded = js_reply.split(',', 1)
    decoded = base64.b64decode(encoded)
    img_array = np.frombuffer(decoded, dtype=np.uint8)
    img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
    return img

def image_to_base64(img):
    _, buffer = cv2.imencode('.jpg', img)
    encoded = base64.b64encode(buffer).decode('utf-8')
    return f"data:image/jpeg;base64,{encoded}"

# 4. メインの推論ループ
print("Webカメラを起動します。ブラウザのカメラ許可ダイアログが出たら許可してください。")
video_stream()

b64_img = ""
while True:
    try:
        # JSからカメラ画像を取得しつつ、前回のアノテーション画像をJSに渡して描画
        js_reply = video_frame(b64_img)

        if not js_reply:
            print("ストリームが停止されました。")
            break

        # Base64文字列をOpenCVの画像配列に変換
        frame = js_to_image(js_reply)

        # YOLOv8で推論 (conf=0.5で信頼度50%以上のものを検出、verbose=Falseでログを抑制)
        results = model.predict(frame, conf=0.5, verbose=False)

        # バウンディングボックスが描画された画像を取得
        annotated_frame = results[0].plot()

        # JSに送り返すためにBase64に変換
        b64_img = image_to_base64(annotated_frame)

    except Exception as e:
        print(f"エラー発生により停止しました: {e}")
        break

Webカメラを起動します。ブラウザのカメラ許可ダイアログが出たら許可してください。


<IPython.core.display.Javascript object>

ストリームが停止されました。


**以上で演習を終了します。お疲れ様でした。**